# ForecastEx

## Web REST API

In [ ]:
import requests, json, os
from pprint import pprint

## Get the current markets

In [ ]:
from get_forecastex_markets import get_forecastex_markets

In [ ]:
fn_market = 'forecastex_markets.json'
if os.path.exists(fn_market):
    with open(fn_market, 'r') as f:
        markets = json.load(f)
else:
    response = requests.get(
        'https://localhost:5000/v1/api/trsrv/event/category-tree',
        verify=False  # skip SSL verification for local gateway
    )
    markets=get_forecastex_markets(response.json())
    with open('forecastex_markets.json', 'w') as f:
        json.dump(markets, f, indent=4)

## Scrape the markets site

In [ ]:
import requests
from collections import defaultdict
market_url = f"https://forecasttrader.interactivebrokers.com/portal.proxy/v1/etp/trsrv/event/contracts"
pricing_url = f"https://forecasttrader.interactivebrokers.com/portal.proxy/v1/etp/trsrv/event/binaryoptions"

In [ ]:
# -- Step 1: Get all contracts under the market
def fetch_all_contracts(market_conid):
    params = {
        "showrestricted": "false",
        "market": market_conid
    }
    r = requests.get(market_url, params=params)
    r.raise_for_status()
    contracts = r.json()["contracts"]
    return contracts

In [ ]:
market_conid = "796056051"

In [ ]:
contracts = fetch_all_contracts(market_conid)

In [ ]:
contracts[0]

In [ ]:
# -- Step 2: Group contracts by candidate with YES and NO
def extract_candidate_conids(contract_data):
    candidates = defaultdict(dict)
    for c in contract_data:
        name = c.get("strikeLabel")
        direction = "YES" if c.get("putOrCall") == "C" else "NO"
        candidates[name][direction] = {
            "conid": c["conid"],
            "description": c["shortDescription"]
        }
    return candidates

In [ ]:
candidates = extract_candidate_conids(contracts)

In [ ]:
candidates

In [ ]:
conids = [side["conid"] for c in candidates.values() for side in c.values()]

In [ ]:
conids

## Getting somewhere

In [ ]:
from get_candidate_probability import get_candidate_probability

In [ ]:
result = get_candidate_probability(796056520)  # Mamdani YES conid
print(f"✅ Mamdani latest probability: {result['probability_pct']:.1f}% (as of {result['timestamp']})")

In [ ]:
conid = 796056520

In [ ]:
period="1week"

In [ ]:
    url = "https://forecasttrader.interactivebrokers.com/tws.proxy/public/hmds/forecastContract"
    params = {
        "conid": conid,
        "period": period,
        "exchange": "FORECASTX",
        "secType": "OPT"
    }

    r = requests.get(url, params=params)
    r.raise_for_status()
    data = r.json()


In [ ]:
[x for x in data]

In [ ]:
result

## Final version get market data

In [ ]:
market = {'conid': 796056051, 'name': 'General Election for New York City Mayor', 'symbol': 'MNYCG'}

### Discover All YES/NO Candidate Subcontracts

In [ ]:
import requests

# Market container for NYC 2025
market_conid = str(market["conid"])
contracts_url = "https://forecasttrader.interactivebrokers.com/portal.proxy/v1/etp/trsrv/event/contracts"
contracts_params = {"showrestricted":"false", "market":market_conid}

contracts_resp = requests.get(contracts_url, params=contracts_params)
contracts_resp.raise_for_status()
contracts_raw = contracts_resp.json()["contracts"]

# Organize by candidate and side
from collections import defaultdict
candidates = defaultdict(dict)
for c in contracts_raw:
    name = c["strikeLabel"]
    side = "YES" if c["putOrCall"] == "C" else "NO"
    candidates[name][side] = c["conid"]

print("CANDIDATES & CONIDs:")
for name, sides in candidates.items():
    print(f"{name:10} YES: {sides['YES']} NO: {sides['NO']}")


###  Get LIVE PRICING for All Candidates

In [ ]:
# Build list of all YES and NO conids
conid_list = []
for sides in candidates.values():
    conid_list.extend([sides["YES"], sides["NO"]])

pricing_url = "https://forecasttrader.interactivebrokers.com/portal.proxy/v1/etp/trsrv/event/binaryoptions"
pricing_params = {"conids": ",".join(str(cid) for cid in conid_list)}

pricing_resp = requests.get(pricing_url, params=pricing_params)
pricing_resp.raise_for_status()
live_data = pricing_resp.json()

# Build a lookup dict by conid
live_lookup = {d["conid"]: d for d in live_data}

### Get LATEST HISTORICAL "LINE CHART" VALUE for Each YES Contract

In [ ]:
def get_latest_probability(conid):
    """Returns the latest YES probability from forecastContract chart, or None if unavailable."""
    url = "https://forecasttrader.interactivebrokers.com/tws.proxy/public/hmds/forecastContract"
    params = {
        "conid": conid,
        "period": "1week",
        "exchange": "FORECASTX",
        "secType": "OPT"
    }
    r = requests.get(url, params=params)
    r.raise_for_status()
    data = r.json()
    avg = data.get("avg")
    if not avg:
        return None
    return round(avg[-1] * 100, 2)

### Print the Full Per-Candidate Table (Current, Bid/Ask, Historical Line)

In [ ]:
print("\nNYC Mayor Election Market — ForecastEx Live Snapshot")
print(f"{'Candidate':10s} {'YES Line%':>9s}")

for name, sides in candidates.items():
    yes = live_lookup.get(sides["YES"], {})
    no  = live_lookup.get(sides["NO"], {})
    # Most recent line chart/YES-probability
    yes_prob = get_latest_probability(sides["YES"])
    line_str = f"{yes_prob:>9.2f}" if yes_prob is not None else "    —    "
    print(f"{name:10s}  "
          f"{line_str}")

In [ ]:
import requests

# The target endpoint
url = "https://forecasttrader.interactivebrokers.com/tws.proxy/public/hmds/forecastContract"

# Query parameters
params = {
    "conid": "796056520",
    "period": "1week",
    "exchange": "FORECASTX",
    "secType": "OPT"
}

# HTTP headers, skipping HTTP/2 pseudo-headers (':authority', ':method', ':path', ':scheme')
headers = {
    "accept": "*/*",
    "accept-encoding": "gzip, deflate, br, zstd",
    "accept-language": "en-US,en;q=0.9",
    "cache-control": "no-cache",
    "content-type": "application/json; charset=utf-8",
    "dnt": "1",
    "pragma": "no-cache",
    "priority": "u=1, i",
    "referer": "https://forecasttrader.interactivebrokers.com/eventtrader/",
    "sec-ch-ua": "\"Google Chrome\";v=\"137\", \"Chromium\";v=\"137\", \"Not/A)Brand\";v=\"24\"",
    "sec-ch-ua-mobile": "?0",
    "sec-ch-ua-platform": "\"Linux\"",
    "sec-fetch-dest": "empty",
    "sec-fetch-mode": "cors",
    "sec-fetch-site": "same-origin",
    "user-agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/137.0.0.0 Safari/537.36",
    "x-ccp-session-id": "undefined",
    "x-client-label": "IB",
    "x-embedded-in": "web",
    "x-request-id": "45",
    "x-service": "AM.LOGIN",
    "x-session-id": "334fa90c-aca9-42ef-8502-72ab677f34b7",
    "x-wa-version": "3aadda4e,Wed, 9 Jul 2025 21:57:03 +0000/2025-07-09T22:02:47.723Z"
}

# Execute the GET request
response = requests.get(url, params=params, headers=headers)

# Print the response
print(response.status_code)

In [ ]:
print(response.json())  # Use .text if the response is not JSON

In [ ]:
import requests

# ---------- CONFIG ----------
base_url = "https://forecasttrader.interactivebrokers.com"
conid = "796056520"
underlyingConid = "796056051"
headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json"
}

# ---------- CANDIDATE ENDPOINTS ----------
endpoints = {
    "contract_market": f"/tws.proxy/public/forecasttrader/contract/market?underlyingConid={underlyingConid}",
    "contract_details": f"/tws.proxy/public/forecasttrader/contract/details?conid={conid}",
    "event_binaryoptions": f"/portal.proxy/v1/etp/trsrv/event/binaryoptions?conids={conid},{int(conid)+5}",
    "event_contracts": f"/portal.proxy/v1/etp/trsrv/event/contracts?showrestricted=false&market={underlyingConid}",
    # For debugging or deeper analysis
    "forecast_contract_timeseries": f"/tws.proxy/public/hmds/forecastContract?conid={conid}&period=1week&exchange=FORECASTX&secType=OPT"
}

# ---------- MAIN LOOP ----------
print("📡 Testing likely endpoints for Open Interest (OI)...\n")

for name, path in endpoints.items():
    full_url = f"{base_url}{path}"
    print(f"➡️  {name} →\n   {full_url}")
    
    try:
        response = requests.get(full_url, headers=headers)
        response.raise_for_status()
        print(f"✅ Success [{response.status_code}]")

        # Pretty print just the top level keys or full JSON
        try:
            json_data = response.json()
            if isinstance(json_data, dict):
                print("🔍 Top-level keys:", list(json_data.keys()))
                pprint(json_data)
            elif isinstance(json_data, list):
                print(f"📦 Response is a list with {len(json_data)} items.")
                pprint(json_data)
            else:
                print("⚠️ Unexpected JSON format.")
        except ValueError:
            print("⚠️ Response is not valid JSON.")

    except Exception as e:
        print(f"❌ Failed: {e}")

    
    
    print("-" * 80)



In [ ]:
sum([30.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            10.0,
            0.0,
            0.0,
            0.0,
            95.0,
            0.0,
            210.0,
            0.0,
            0.0,
            0.0,
            290.0,
            0.0,
            0.0,
            0.0,
            1984.0,
            0.0,
            0.0,
            2300.0,
            5.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            1.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            845.0,
            0.0,
            460.0,
            621.0,
            6.0,
            14.0,
            504.0,
            88.0,
            37.0,
            1.0,
            0.0,
            0.0,
            0.0,
            0.0,
            1269.0,
            0.0,
            5.0,
            0.0,
            0.0,
            4.0,
            2000.0,
            0.0,
            6.0,
            100.0,
            0.0,
            0.0,
            5.0,
            0.0,
            0.0,
            11.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            5.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            90.0,
            0.0,
            0.0,
            0.0,
            0.0,
            100.0,
            0.0,
            0.0,
            0.0,
            10.0,
            100.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            10000.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            604.0,
            0.0,
            0.0,
            0.0,
            0.0,
            3.0,
            17.0,
            83.0,
            100.0,
            0.0,
            0.0,
            103.0,
            0.0,
            1000.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            1000.0,
            100.0,
            0.0,
            0.0,
            1390.0,
            20.0,
            0.0,
            350.0,
            0.0,
            0.0,
            1.0,
            0.0,
            100.0,
            100.0,
            12.0,
            100.0,
            880.0,
            100.0,
            0.0,
            100.0,
            3034.0,
            7.0,
            0.0])

In [ ]:
import requests

url = "https://forecasttrader.interactivebrokers.com/tws.proxy/public/forecasttrader/category/tree"
headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json",
}

try:
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    json_data = response.json()
    print("JSON response:")
    print(json_data)
except Exception as e:
    print("Error during request:", e)


In [ ]:
import requests

url = "https://forecasttrader.interactivebrokers.com/portal.proxy/v1/etp/trsrv/secdef"

payload = {
    "conids": ["796056520", "796056525"],
    "contracts": False
}

#response = requests.post(url, headers=headers, json=payload)
response = requests.post(url, headers={}, json=payload)
# Ensure it worked
if response.ok:
    try:
        data = response.json()
        print("📦 Response JSON:")
        pprint(data)
    except ValueError:
        print("⚠️ Response was not JSON:")
        print(response.text)
else:
    print(f"❌ Request failed: {response.status_code}")
    print(response.text)


In [ ]:
import requests

url = "https://forecasttrader.interactivebrokers.com/tws.proxy/public/hmds/scanner/forecastOpenInterest"

headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json",
    "Referer": "https://forecasttrader.interactivebrokers.com/eventtrader/"
}

response = requests.get(url, headers=headers)

if response.ok:
    try:
        data = response.json()
        print(f"✅ Retrieved {len(data)} results.")
        for item in data[:5]:  # Show top 5 entries
            print(item)
    except ValueError:
        print("❌ Failed to parse JSON.")
else:
    print(f"❌ Request failed with status code {response.status_code}")
    print(response.text)


In [ ]:
pprint(data)

https://www.perplexity.ai/search/how-do-i-do-this-request-in-py-LKWOZo0lRNegn9r7ALA1dw

In [ ]:
import websocket
import threading
import json

# ✅ WebSocket endpoint for ForecastTrader
WS_URL = "wss://forecasttrader.interactivebrokers.com/portal.proxy/v1/etp/ws"

# ✅ Contract ID to monitor (example from your trace)
CONTRACT_ID = "796056520"

# ✅ Open Interest = field 7638
FIELDS = ["7638"]

def on_open(ws):
    print("✅ WebSocket connected! Subscribing to contract data...")
    msg = f'smd+{CONTRACT_ID}+{json.dumps({"fields": FIELDS, "backout": True})}'
    ws.send(msg)

def on_message(ws, message):
    try:
        data = json.loads(message)
        if "7638" in data:
            print(f"📊 Open Interest for conid {data.get('conid', 'unknown')}: {data['7638']}")
        else:
            print("💬 Non-OI message:", data)
    except Exception as e:
        print("🛑 Error parsing message:", e)
        print("Raw message:", message)

def on_error(ws, error):
    print(f"❌ WebSocket error: {error}")

def on_close(ws, status_code, msg):
    print(f"🔌 WebSocket closed: {status_code} — {msg}")

# ✅ Use cookies captured from working `forecasttrader.interactivebrokers.com` browser session
cookie_header = (
    "SBID=ua29hrciu7fmah5ej6o; "
    "device.info=eyJpZCI6ImI0OGJjMmUwIiwibWFjIjoiMTY6REI6OUY6RjE6NkM6QTIifQ==; "
    "IB_PRIV_PREFS=0%7C0%7C0; "
    "web=3311011730; "
    "IB_LANG=en; "
    "ET_UUID=e5fb05f2-c4d6-4cbd-bb1c-eb315ffc1d55; "
    "NONAUTH_AB_UUID=ab324c61-ac4c-4045-da9e-527494f764e1; "
    "ib=googlead; "
    "_fbp=fb.1.1753132383798.361859181467435267; "
    "_ga_7FYT07EYXG=GS2.1.s1753133116$o2$g0$t1753133116$j60$l0$h0; "
    "sm_uuid=1753136256764; "
    "_tt_enable_cookie=1; "
    "_ttp=01K0QGV8ZS15M7K0Q8FRZ1TQMH_.tt.1; "
    "_ga=GA1.1.1895860845.1753136080; "
    "Campus_tag_ga=GA1.1.2102125515.1753136746; "
    "RT=\"z=1&dm=forecasttrader.interactivebrokers.com&si=4da27d71-25d6-4a10-b266-1e7f75766ab0&ss=mddzsnvr&sl=0&tt=0\"; "
    "Campus_tag_ga_3DZW8R5ZLR=GS2.1.s1753158810$o5$g1$t1753159153$j60$l0$h0; "
    "URL_PARAM=\"RL=1&locale=en_US\"; "
    "RT=\"z=1&dm=interactivebrokers.com&si=acfbbac5-d75c-44e1-8bbd-3135ad1a3244&ss=mdf3rp7m&sl=1&tt=1dp&rl=1&ld=1gh&ul=8dw&hd=8vg\"; "
    "etp=cdb9bc9a597eb700820f549ec0ae3f43; "
    "PHPSESSID=p3n903ie4prka117q72jc98pem; "
    "dc=use1; "
    "AKA_A2=A; "
    "_rdt_uuid=1753065529790.79ce9976-6b8d-45c2-ad31-2c847e0ad115; "
    "ttcsid_CCH57TBC77U4E617QA80=1753231052660::jVG8d7B5FyNfhaxKJK7B.7.1753231053959; "
    "ttcsid=1753231052660::DOG3IVHJUCuWSQez-K-m.7.1753231053959; "
    "_ga_FFHP0Y799T=GS2.1.s1753231725$o9$g0$t1753231725$j60$l0$h0"
)

# ✅ Full headers to simulate an authenticated ForecastTrader browser client
headers = [
    "Origin: https://forecasttrader.interactivebrokers.com",
    "Referer: https://forecasttrader.interactivebrokers.com/en/home.php",
    "User-Agent: Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/137.0.0.0 Safari/537.36",
    f"Cookie: {cookie_header}"
]

# ✅ Setup WebSocket client
ws_app = websocket.WebSocketApp(
    WS_URL,
    header=headers,
    on_open=on_open,
    on_message=on_message,
    on_error=on_error,
    on_close=on_close
)

# 🔁 Start connection in background
thread = threading.Thread(target=ws_app.run_forever)
thread.start()


In [ ]:
import websocket
import threading
import json
import time

# WebSocket endpoint for ForecastTrader
WS_URL = "wss://forecasttrader.interactivebrokers.com/portal.proxy/v1/etp/ws"

# Open Interest = field 7638
FIELDS = ["7638"]

# Use cookies captured from a live ForecastTrader browser session
cookie_header = (
    "SBID=ua29hrciu7fmah5ej6o; "
    "device.info=eyJpZCI6ImI0OGJjMmUwIiwibWFjIjoiMTY6REI6OUY6RjE6NkM6QTIifQ==; "
    "IB_PRIV_PREFS=0%7C0%7C0; "
    "web=3311011730; "
    "IB_LANG=en; "
    "ET_UUID=e5fb05f2-c4d6-4cbd-bb1c-eb315ffc1d55; "
    "NONAUTH_AB_UUID=ab324c61-ac4c-4045-da9e-527494f764e1; "
    "ib=googlead; "
    "_fbp=fb.1.1753132383798.361859181467435267; "
    "_ga_7FYT07EYXG=GS2.1.s1753133116$o2$g0$t1753133116$j60$l0$h0; "
    "sm_uuid=1753136256764; "
    "_tt_enable_cookie=1; "
    "_ttp=01K0QGV8ZS15M7K0Q8FRZ1TQMH_.tt.1; "
    "_ga=GA1.1.1895860845.1753136080; "
    "Campus_tag_ga=GA1.1.2102125515.1753136746; "
    "RT=\"z=1&dm=forecasttrader.interactivebrokers.com&si=4da27d71-25d6-4a10-b266-1e7f75766ab0&ss=mddzsnvr&sl=0&tt=0\"; "
    "Campus_tag_ga_3DZW8R5ZLR=GS2.1.s1753158810$o5$g1$t1753159153$j60$l0$h0; "
    "URL_PARAM=\"RL=1&locale=en_US\"; "
    "RT=\"z=1&dm=interactivebrokers.com&si=acfbbac5-d75c-44e1-8bbd-3135ad1a3244&ss=mdf3rp7m&sl=1&tt=1dp&rl=1&ld=1gh&ul=8dw&hd=8vg\"; "
    "etp=cdb9bc9a597eb700820f549ec0ae3f43; "
    "PHPSESSID=p3n903ie4prka117q72jc98pem; "
    "dc=use1; "
    "AKA_A2=A; "
    "_rdt_uuid=1753065529790.79ce9976-6b8d-45c2-ad31-2c847e0ad115; "
    "ttcsid_CCH57TBC77U4E617QA80=1753231052660::jVG8d7B5FyNfhaxKJK7B.7.1753231053959; "
    "ttcsid=1753231052660::DOG3IVHJUCuWSQez-K-m.7.1753231053959; "
    "_ga_FFHP0Y799T=GS2.1.s1753231725$o9$g0$t1753231725$j60$l0$h0"
)

headers = [
    "Origin: https://forecasttrader.interactivebrokers.com",
    "Referer: https://forecasttrader.interactivebrokers.com/en/home.php",
    "User-Agent: Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/137.0.0.0 Safari/537.36",
    f"Cookie: {cookie_header}"
]

def get_OI_for_conids(conids):
    """
    Get Open Interest (OI) for a list of contract IDs (conids) from ForecastTrader.
    Returns a dictionary: {conid (str): OI (str or numeric)}
    """
    oi_results = {}
    received_conids = set()
    lock = threading.Lock()
    done_event = threading.Event()

    def on_open(ws):
        # Subscribe for all requested conids
        for conid in conids:
            msg = f'smd+{conid}+{json.dumps({"fields": FIELDS, "backout": True})}'
            ws.send(msg)

    def on_message(ws, message):
        try:
            data = json.loads(message)
            if "7638" in data and "conid" in data:
                conid = str(data["conid"])
                oi = data["7638"]
                with lock:
                    if conid not in received_conids:
                        oi_results[conid] = oi
                        received_conids.add(conid)
                    if len(received_conids) == len(conids):
                        done_event.set()
                        ws.close()
            # (optional) print non-OI messages if wanted
        except Exception:
            pass  # Ignore unparseable messages

    def on_error(ws, error):
        done_event.set()

    def on_close(ws, code, msg):
        done_event.set()

    ws_app = websocket.WebSocketApp(
        WS_URL,
        header=headers,
        on_open=on_open,
        on_message=on_message,
        on_error=on_error,
        on_close=on_close,
    )

    thread = threading.Thread(target=ws_app.run_forever)
    thread.daemon = True
    thread.start()
    # Wait for result or timeout
    done_event.wait(timeout=20)
    if ws_app.keep_running:
        ws_app.close()
    return oi_results

In [ ]:
get_OI_for_conids(["796056520", "796056525"])

In [1]:
import asyncio
import websockets
import json
import requests

def get_websocket_debugger_url(domain="forecasttrader.interactivebrokers.com", port=9222):
    tabs = requests.get(f"http://localhost:{port}/json").json()
    for tab in tabs:
        if domain in tab.get("url", ""):
            return tab["webSocketDebuggerUrl"]
    raise Exception("ForecastTrader tab not found. Is it open?")

async def get_cookie_header_from_devtools():
    ws_url = get_websocket_debugger_url()
    async with websockets.connect(ws_url) as ws:
        await ws.send(json.dumps({
            "id": 1,
            "method": "Network.enable"
        }))
        await ws.recv()  # Ack

        await ws.send(json.dumps({
            "id": 2,
            "method": "Network.getAllCookies"
        }))

        while True:
            response = await ws.recv()
            message = json.loads(response)
            if "id" in message and message["id"] == 2:
                cookies = message.get("result", {}).get("cookies", [])
                filtered = [
                    f"{c['name']}={c['value']}"
                    for c in cookies
                    if "interactivebrokers.com" in c['domain']
                ]
                header = "; ".join(filtered)
                return header

# Usage in a live notebook or script:
async def main():
    cookie_header = await get_cookie_header_from_devtools()
    print("✅ Final cookie header:\n")
    print(cookie_header)

await main()


✅ Final cookie header:

AKA_A2=A; ET_UUID=33b8417e-1980-4a9d-95e9-00a674a69060; etp=4c8227803c69848a06a5943e1701ace4; _rdt_uuid=1753237684083.1946c47d-18d1-4354-90bd-8a31db0c7afa; sm_uuid=1753238219585; _ga=GA1.1.298830904.1753237684; _ga_FFHP0Y799T=GS2.1.s1753237684$o1$g0$t1753237684$j60$l0$h0; _tt_enable_cookie=1; _ttp=01K0THR033VX39BDC01AADVG2A_.tt.1; ttcsid=1753237684326::rjnyNrnXn7biHov38HMo.1.1753237684326; _fbp=fb.1.1753237684361.490110541229587160; ttcsid_CCH57TBC77U4E617QA80=1753237684325::Tlq13y43f7vPe0nxIJDu.1.1753237684549


In [2]:
import websocket
import threading
import json
import asyncio
import requests
import websockets

# --- Step 1: Auto-fetch cookies from Chrome's DevTools session ---
def get_websocket_debugger_url(domain="forecasttrader.interactivebrokers.com", port=9222):
    tabs = requests.get(f"http://localhost:{port}/json").json()
    for tab in tabs:
        if domain in tab.get("url", ""):
            return tab["webSocketDebuggerUrl"]
    raise Exception("ForecastTrader tab not found.")

async def get_cookie_header_from_chrome_devtools():
    ws_url = get_websocket_debugger_url()
    async with websockets.connect(ws_url) as ws:
        await ws.send(json.dumps({"id": 1, "method": "Network.enable"}))
        await ws.recv()  # Discard ack

        await ws.send(json.dumps({"id": 2, "method": "Network.getAllCookies"}))

        while True:
            response = await ws.recv()
            message = json.loads(response)
            if message.get("id") == 2:
                cookies = message["result"]["cookies"]
                relevant = [
                    f"{c['name']}={c['value']}"
                    for c in cookies
                    if "interactivebrokers.com" in c["domain"]
                ]
                return "; ".join(relevant)

# --- Step 2: Get OI Mapping via WebSocket ---
def get_OI_for_conids(conids, cookie_header):
    WS_URL = "wss://forecasttrader.interactivebrokers.com/portal.proxy/v1/etp/ws"
    FIELDS = ["7638"]
    oi_results = {}
    received_conids = set()
    lock = threading.Lock()
    done_event = threading.Event()

    # Prepare headers
    headers = [
        "Origin: https://forecasttrader.interactivebrokers.com",
        "Referer: https://forecasttrader.interactivebrokers.com/en/home.php",
        "User-Agent: Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/137.0.0.0 Safari/537.36",
        f"Cookie: {cookie_header}"
    ]

    def on_open(ws):
        for conid in conids:
            msg = f'smd+{conid}+{json.dumps({"fields": FIELDS, "backout": True})}'
            ws.send(msg)

    def on_message(ws, message):
        try:
            data = json.loads(message)
            if "7638" in data and "conid" in data:
                conid = str(data["conid"])
                oi = data["7638"]
                with lock:
                    if conid not in received_conids:
                        oi_results[conid] = oi
                        received_conids.add(conid)
                    if len(received_conids) >= len(conids):
                        done_event.set()
                        ws.close()
        except Exception:
            pass

    def on_error(ws, error):
        print("❌ WebSocket error:", error)
        done_event.set()

    def on_close(ws, code, msg):
        done_event.set()

    ws_app = websocket.WebSocketApp(
        WS_URL,
        header=headers,
        on_open=on_open,
        on_message=on_message,
        on_error=on_error,
        on_close=on_close
    )

    thread = threading.Thread(target=ws_app.run_forever)
    thread.daemon = True
    thread.start()

    done_event.wait(timeout=20)
    if ws_app.keep_running:
        ws_app.close()
    return oi_results

# --- Step 3: Async wrapper to run it all ---
async def run_get_OI(conids):
    cookie_header = await get_cookie_header_from_chrome_devtools()
    result = get_OI_for_conids(conids, cookie_header)
    print("\n✅ Final Result:\n", result)
    return result

# --- Example usage ---
# Run this in an async environment like Jupyter or an asyncio-compatible main
await run_get_OI(["796056520", "796056525"])



✅ Final Result:
 {'796056520': '1.16M', '796056525': '1.16M'}


{'796056520': '1.16M', '796056525': '1.16M'}